In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.types import *
from pyspark.sql import SparkSession
import pandas as pd

In [ ]:
spark = (
    SparkSession.builder
      .config("spark.driver.memory", "48g")                 # ajuste p/ 16g–24g
      .config("spark.sql.execution.arrow.pyspark.enabled", "true")
      .config("spark.sql.execution.arrow.maxRecordsPerBatch", "20000")
      .config("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)  # 64MB/partição
      .config("spark.driver.maxResultSize", "0")            # sem limite de resultado (cuidado)
      .getOrCreate()
)

spark.version

In [ ]:
# read table
data = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/dados/consultas.parquet"
)
data.count()

In [ ]:
# filter pacientes in target data
pacientes_target = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/target_internacao.parquet"
).select("prontuario").distinct()

data = data.join(pacientes_target, on="prontuario", how="inner")
data.count()

In [ ]:
# count nulls in "descricao_anamnese"
data.select(F.sum(F.col("descricao_anamnese").isNull().cast("int")).alias("nulls_descricao_anamnese")).show()

In [ ]:
data.show(10)

In [ ]:
# converte as strings para timestamp
data = data.withColumn("dt_consulta", F.to_timestamp("dt_consulta")) 

In [ ]:
# filter time
data = data.filter((F.col("dt_consulta") >= F.lit("2020-08-01")) &( F.col("dt_consulta") <= F.lit("2025-10-01")))

In [ ]:
def generate_consulta_features(data, ref_ts):

    # primeiro periodo
    data_filtered1 = data.filter(
    F.col("dt_consulta") >= F.add_months(F.lit(ref_ts), -3)
    )
    agg1 = data_filtered1.select("prontuario").groupBy("prontuario").count()
    agg1 = agg1.withColumnRenamed("count", "qtd_consulta_3_0_m")

    # segundo periodo
    data_filtered2 = data.filter(
    (    F.col("dt_consulta") >= F.add_months(F.lit(ref_ts), -6)) &
    (F.col("dt_consulta") <= F.add_months(F.lit(ref_ts), -3))
    )
    agg2 = data_filtered2.select("prontuario").groupBy("prontuario").count()
    agg2 = agg2.withColumnRenamed("count", "qtd_consultas_6_3_m")

    # terceiro periodo
    data_filtered3 = data.filter(
    (    F.col("dt_consulta") >= F.add_months(F.lit(ref_ts), -9)) &
    (F.col("dt_consulta") <= F.add_months(F.lit(ref_ts), -6))
    )
    agg3 = data_filtered3.select("prontuario").groupBy("prontuario").count()
    agg3 = agg3.withColumnRenamed("count", "qtd_consulta_9_6_m")

    # quarto periodo
    data_filtered4 = data.filter(
    (    F.col("dt_consulta") >= F.add_months(F.lit(ref_ts), -12)) &
    (F.col("dt_consulta") <= F.add_months(F.lit(ref_ts), -9))
    )
    agg4 = data_filtered4.select("prontuario").groupBy("prontuario").count()
    agg4 = agg4.withColumnRenamed("count", "qtd_consultas_12_9_m")

    # join all features
    features = agg1.join(agg2, on="prontuario", how="outer") \
                   .join(agg3, on="prontuario", how="outer") \
                   .join(agg4, on="prontuario", how="outer")
    
    features = features.fillna(0)

    return features

In [ ]:

# construir meses de referencia a partir das datas minima e maxima
bounds = data.agg(
    F.date_trunc("month", F.min("dt_consulta")).alias("min_m"),
    F.date_trunc("month", F.max("dt_consulta")).alias("max_m"),
)

ref_dates_data = bounds.select(
    F.expr("sequence(min_m, max_m, interval 1 month) as ref_dates")
).select(F.explode("ref_dates").alias("ref_date"))

# lista de ref dates distintos
ref_dates = [r.ref_date for r in ref_dates_data.collect()]
len(ref_dates)

In [ ]:
ref_dates

In [ ]:
# define dataframe para incorporar dados ao cursor
features = spark.createDataFrame([], schema=StructType())

# para cada data de referencia, filtrar os dados anteriores a ela, para calculo das variaveis
for ref_ts in ref_dates:

    print(ref_ts)
    
    features_ref_date = generate_consulta_features(data, ref_ts)

    features_ref_date = features_ref_date.withColumn("date_ref", F.lit(ref_ts).cast("timestamp"))

    features = features.unionByName(features_ref_date, allowMissingColumns=True)

In [ ]:
#features.show(10)

In [ ]:
# filter pacientes e date_ref in target data
pacientes_target = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/target_internacao.parquet"
).select("prontuario", "date_ref").distinct()

pacientes_target.count()

In [ ]:
features = features.join(pacientes_target, on=["prontuario", "date_ref"], how="inner")
features.count()

In [ ]:
features.printSchema()

In [ ]:
features.toPandas().to_parquet("C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/features_consultas.parquet")

In [ ]:
# total rows (avoid recomputing repeatedly) 
total_rows = features.count()

# count nulls and compute percentage 
null_stats = features.select([ F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in features.columns ]) 

# add percentages 
null_stats_percent = null_stats.select([ F.col(c).alias(c + "_nulls") for c in features.columns ] + [ (F.col(c) / total_rows * 100).alias(c + "_pct") for c in features.columns ]) 

null_stats_percent.show()